# Visão Computacional na Prática
### Como a CNN enxerga (camada por camada) + Reconhecimento Facial (OpenCV) + Detecção de Objetos (YOLO)

Este notebook roda **inteiramente no Google Colab** — não precisa de GitHub, não precisa instalar nada na sua máquina.

**Como usar:**
1. Vá em `Ambiente de execução` → `Executar tudo` (ou execute célula por célula com `Shift + Enter`)
2. Quando pedir para enviar uma foto, você pode subir uma foto sua, de amigos ou qualquer imagem com pessoas/objetos
3. Leia os comentários no código — eles explicam o que está acontecendo em cada linha

> Tudo que vemos aqui é uma aplicação prática do que vimos em aula sobre CNNs: os modelos abaixo (Haar Cascade e YOLO) foram treinados usando exatamente essa lógica de filtros aprendendo padrões em camadas.

## Parte 1 — Abrindo a "caixa-preta": como a CNN enxerga camada por camada

Até agora vimos os modelos prontos funcionando (Haar Cascade, YOLO), mas sem ver o que acontece **por dentro**.

Aqui vamos usar uma CNN clássica chamada **VGG16**, já treinada para reconhecer 1000 categorias de objetos diferentes, e observar **o que cada camada "enxerga"** conforme a imagem passa por ela — exatamente a hierarquia que vimos em aula: bordas → texturas/formas → partes de objeto → objeto completo.

In [ ]:
import urllib.request
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
import numpy as np

# Carrega a MobileNetV2 já treinada (leve, ~14MB — baixa rápido mesmo em wifi de escola)
modelo_cnn = MobileNetV2(weights='imagenet')

print("Modelo carregado! Ela tem", len(modelo_cnn.layers), "camadas no total.")


In [ ]:
# Escolhemos 5 camadas espalhadas ao longo da rede: da mais rasa (perto da entrada)
# até a mais profunda (perto da decisão final)
camadas_para_ver = ['Conv1', 'block_2_expand', 'block_5_expand', 'block_9_expand', 'block_16_project']

def ver_camadas_da_cnn(caminho_imagem, n_filtros=6):
    """Mostra a imagem original, os mapas de características de cada camada
    escolhida, e a classificação final da rede."""

    # A MobileNetV2 espera imagens 224x224
    img = load_img(caminho_imagem, target_size=(224, 224))
    array_img = img_to_array(img)
    lote = np.expand_dims(array_img, axis=0)
    lote_preparado = preprocess_input(lote.copy())

    # Cria um modelo "espião": em vez de só olhar a resposta final,
    # ele devolve a saída (ativação) de cada camada escolhida
    saidas_das_camadas = [modelo_cnn.get_layer(nome).output for nome in camadas_para_ver]
    modelo_espiao = Model(inputs=modelo_cnn.input, outputs=saidas_das_camadas)
    ativacoes = modelo_espiao.predict(lote_preparado, verbose=0)

    # Mostra a imagem original
    plt.figure(figsize=(3, 3))
    plt.imshow(array_img.astype('uint8'))
    plt.title("Imagem original")
    plt.axis('off')
    plt.show()

    # Para cada camada, mostra alguns dos mapas de características (feature maps)
    for nome_camada, ativacao in zip(camadas_para_ver, ativacoes):
        tamanho = ativacao.shape[1]
        n_filtros_total = ativacao.shape[-1]

        fig, eixos = plt.subplots(1, n_filtros, figsize=(15, 3))
        fig.suptitle(
            f'Camada "{nome_camada}"  →  imagem reduzida para {tamanho}x{tamanho}px, '
            f'com {n_filtros_total} filtros diferentes rodando (mostrando {n_filtros} deles)'
        )
        for i in range(n_filtros):
            mapa = ativacao[0, :, :, i]
            eixos[i].imshow(mapa, cmap='viridis')
            eixos[i].axis('off')
        plt.show()

    # Classificação final: a "decisão" da rede, depois de passar por todas as camadas
    predicao = modelo_cnn.predict(lote_preparado, verbose=0)
    top3 = decode_predictions(predicao, top=3)[0]

    print("A rede acha que isso é:")
    for _, nome_classe, confianca in top3:
        print(f"  - {nome_classe}: {confianca * 100:.1f}%")


# Imagem de exemplo (amostra oficial do TensorFlow, usada em tutoriais)
url_exemplo = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"
urllib.request.urlretrieve(url_exemplo, "exemplo_cnn.jpg")

ver_camadas_da_cnn("exemplo_cnn.jpg")


## Por que a imagem "some" conforme desce nas camadas?

Repare que a **camada 1 parece muito com a foto original**, mas a **última camada parece só ruído abstrato**. Isso não é erro — é assim que uma CNN funciona, e o motivo é simples:

Cada camada faz duas coisas ao mesmo tempo:
1. **Olha para uma área cada vez maior** da imagem original (na primeira camada, cada "pixel" do mapa vem de uma área pequena de ~3x3 pixels; nas últimas camadas, cada "pixel" já resume uma fatia enorme da imagem, por causa do pooling)
2. **Combina o que a camada anterior já encontrou**, ao invés de olhar pixels crus

O resultado é uma troca: a rede vai **perdendo a noção de "onde" exatamente** está cada detalhe, mas **ganhando a noção de "o quê"** esse detalhe significa.

| Camada | O que ela enxerga | Por que ela ainda "parece" a imagem (ou não) |
|---|---|---|
| `Conv1` (primeira) | Bordas e linhas simples | Cada "pixel" do mapa vem de uma área pequena e específica da foto — ainda dá pra reconhecer a forma |
| `block_5_expand`, `block_9_expand` (meio) | Texturas e formas (combinações de bordas) | O mapa já foi encolhido várias vezes (pooling), cada ponto resume uma área maior — começa a perder o "onde" |
| `block_16_project` (última) | Conceitos abstratos (combinação de milhares de padrões anteriores) | Cada ponto resume uma fatia enorme da imagem original — não existe mais correspondência visual direta com os pixels |

**Uma analogia que ajuda:** é como resumir um livro em etapas.
- Camada 1 = ler linha por linha (você vê tudo, bem literal)
- Camadas do meio = o resumo de cada parágrafo (perdeu o texto exato, ganhou a ideia)
- Última camada = o livro inteiro resumido numa frase só (não parece mais texto original, só a essência condensada — mas é exatamente essa frase que permite dizer do que o livro trata)

Para a rede, essa "frase resumida" da última camada é extremamente informativa — é ela que é usada para decidir a classificação final. Para o nosso olho, parece só ruído, porque não fomos treinados para "ler" esse tipo de informação abstrata.

**Reforçando o padrão que você vai ver nos gráficos abaixo:**
- Nas primeiras camadas (`Conv1`), os mapas parecem contornos e bordas simples — ainda muito parecidos com a imagem original
- Nas camadas do meio (`block_5_expand`, `block_9_expand`), já aparecem texturas e formas mais abstratas
- Na última camada (`block_16_project`), os mapas já não se parecem em nada com a imagem original a olho nu — é pura informação abstrata que a rede usa para decidir a classificação final
- A imagem também vai **diminuindo de tamanho** a cada bloco — efeito do pooling/strides que vimos em aula

### Agora com a SUA imagem


In [ ]:
from google.colab import files

enviados = files.upload()

for nome_arquivo in enviados.keys():
    print(f"\nProcessando: {nome_arquivo}")
    ver_camadas_da_cnn(nome_arquivo)


## Preparação para as próximas partes (rode antes de continuar)

Rode a célula abaixo uma vez. Ela instala:
- **opencv-python-headless**: a biblioteca OpenCV (a versão "headless" é a recomendada para rodar em notebooks, sem interface gráfica)
- **ultralytics**: biblioteca oficial do YOLO


In [ ]:
!pip install -q opencv-python-headless ultralytics matplotlib

import cv2
import matplotlib.pyplot as plt
import urllib.request

print("Bibliotecas instaladas e prontas!")


## Parte 2 — Reconhecimento Facial com OpenCV

Vamos usar um **Haar Cascade**, um classificador clássico que já vem pronto dentro do próprio OpenCV, treinado para detectar rostos.

Relembrando da aula: o classificador varre a imagem procurando o padrão visual de um rosto (regiões de contraste entre olhos, nariz, boca), de forma parecida com os filtros de uma CNN.

In [ ]:
def detectar_rostos(caminho_imagem):
    """Recebe o caminho de uma imagem, detecta rostos e mostra o resultado."""

    # Carrega o classificador de rostos que já vem pronto dentro do OpenCV
    caminho_cascade = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    face_cascade = cv2.CascadeClassifier(caminho_cascade)

    # Carrega a imagem e converte para escala de cinza
    # (o Haar Cascade trabalha em escala de cinza, não precisa de cor para achar rostos)
    imagem = cv2.imread(caminho_imagem)
    imagem_cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

    # Detecta os rostos na imagem
    # scaleFactor: o quanto a imagem é reduzida a cada tentativa de detecção
    # minNeighbors: quantas detecções vizinhas são necessárias para confirmar um rosto
    rostos = face_cascade.detectMultiScale(imagem_cinza, scaleFactor=1.1, minNeighbors=5)

    print(f"Rostos detectados: {len(rostos)}")

    # Desenha um retângulo verde ao redor de cada rosto encontrado
    for (x, y, largura, altura) in rostos:
        cv2.rectangle(imagem, (x, y), (x + largura, y + altura), (0, 255, 0), 3)

    # OpenCV usa a ordem de cores BGR, e o matplotlib usa RGB — precisa converter para exibir corretamente
    imagem_rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(imagem_rgb)
    plt.axis('off')
    plt.show()


# Imagem de exemplo (amostra oficial de tutoriais do OpenCV)
url_exemplo = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/messi5.jpg"
urllib.request.urlretrieve(url_exemplo, "exemplo_rosto.jpg")

detectar_rostos("exemplo_rosto.jpg")


### Agora com a SUA foto

Rode a célula abaixo, clique em **"Escolher arquivos"** e envie uma foto do seu computador (sua, de amigos, da família — qualquer imagem com rostos).

In [ ]:
from google.colab import files

enviados = files.upload()

# Roda a detecção de rosto em cada imagem enviada
for nome_arquivo in enviados.keys():
    print(f"\nProcessando: {nome_arquivo}")
    detectar_rostos(nome_arquivo)


## Parte 3 — Detecção de Objetos com YOLO

Agora vamos usar o **YOLOv8** (You Only Look Once), o algoritmo que vimos em aula para detecção de objetos em tempo real.

Na primeira execução, o modelo pré-treinado (`yolov8n.pt` — a versão "nano", mais leve e rápida) é baixado automaticamente.

In [ ]:
from ultralytics import YOLO

# Carrega o modelo YOLOv8 pré-treinado (já sabe reconhecer 80 tipos de objetos: pessoa, carro, cachorro, celular, etc.)
modelo_yolo = YOLO('yolov8n.pt')

def detectar_objetos(caminho_imagem):
    """Recebe o caminho de uma imagem, detecta objetos e mostra o resultado."""

    resultados = modelo_yolo(caminho_imagem)

    # .plot() já desenha as caixas, os nomes das classes e a confiança de cada detecção
    imagem_com_caixas = resultados[0].plot()

    # Convertendo de BGR (OpenCV) para RGB (matplotlib)
    imagem_rgb = cv2.cvtColor(imagem_com_caixas, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 10))
    plt.imshow(imagem_rgb)
    plt.axis('off')
    plt.show()

    # Lista no console tudo que foi detectado, com o nome da classe e a confiança
    print("Objetos detectados:")
    for caixa in resultados[0].boxes:
        classe = modelo_yolo.names[int(caixa.cls[0])]
        confianca = float(caixa.conf[0]) * 100
        print(f"  - {classe}: {confianca:.1f}% de confiança")


# Imagem de exemplo (amostra oficial da Ultralytics, criadora do YOLO)
url_exemplo = "https://ultralytics.com/images/bus.jpg"
urllib.request.urlretrieve(url_exemplo, "exemplo_objetos.jpg")

detectar_objetos("exemplo_objetos.jpg")


### Agora com a SUA imagem

Envie uma foto de uma rua, sala, mesa com vários objetos — quanto mais coisas na imagem, mais interessante fica o resultado!

In [ ]:
from google.colab import files

enviados = files.upload()

for nome_arquivo in enviados.keys():
    print(f"\nProcessando: {nome_arquivo}")
    detectar_objetos(nome_arquivo)


## Desafio (opcional)

Agora que você viu os dois modelos funcionando, tente:

1. Rodar a detecção de rostos numa foto em grupo (várias pessoas) — ele encontra todos?
2. No código do YOLO, troque `yolov8n.pt` por `yolov8s.pt` (versão um pouco maior) e compare o resultado e o tempo de execução
3. Teste uma imagem "difícil": pouca luz, ângulo estranho, objeto parcialmente escondido — o modelo ainda acerta?

Isso ajuda a perceber, na prática, os limites desses modelos — algo que muitas vezes passa despercebido só na teoria.